### **Practical Assignment - Machine Learning I**
##### Work done by:

Enrico Sanchez, upXXXXXXXXX

Enzo Grigorio, up202404041

Matheus Guerra, upXXXXXXXXX

Implementation based on the repository [Rushter/MLAlgorithms](https://github.com/rushter/MLAlgorithms/), as permitted by the assignement instructions.

### Hypothesis (Phase 1)
The **k-NN algorithm** is an instance-based learning method that determines a point's class based on the simple majority of its nearest neighbors. Our hypothesis is that it exhibits low robustness to **Dataset Group 1 (Noise/Outliers)** due to the following factors:

* **Local Sensitivity**: With a low $k$ value, a single incorrectly positioned outlier can "capture" the neighborhood of legitimate points, forcing a misclassification.
* **Distance Metric Issues**: Since the standard (unweighted) k-NN treats all neighbors equally, an outlier has the same influence on the decision as a point that is much closer and more representative of the class.
* **Decision Boundary Distortion**: The presence of noise creates "islands" of incorrect classes in the feature space, fragmenting the decision boundary and hindering the model's ability to generalize.

This evaluation is part of **Phase 1**, where we aim to understand how these data characteristics affect the standard version of the algorithm before proposing a more robust variant in **Phase 2**.

#### ***Imports***

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

from knn import KNNClassifier

### ***Loading the datasets***

In [ ]:
folder_name = 'noise_outliers'

selected_datasets = [
    '31_credit-g.csv',
    '451_irish.csv',
    '56_vote.csv',
    '27_colic.csv',
    '745_autoprice.csv',
    '38_sick.csv',
    '786_cleveland.csv',
    '50_tic-tac-toe.csv',
    '13_breast-cancer.csv',
    '466_schizo.csv'
]

data_registry = {}
#this loop will iterate through the datasets to show the original state, before any normalization
for ds_name in selected_datasets:
    path = os.path.join(folder_name, ds_name)
    df_raw = pd.read_csv(path)
    print(f"\n Dataset: {ds_name} (Original State):")
    categorical_cols = df_raw.select_dtypes(include=['object']).columns.tolist()
    print(f"Categorical columns to convert: {categorical_cols}")
    display(df_raw.head())
#this one will iterate and convert any string column to numbers
for ds_name in selected_datasets:
    path = os.path.join(folder_name, ds_name)
    df = pd.read_csv(path)
    df = df.dropna()

    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].astype('category').cat.codes

    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values
    X_min = X.min(axis=0)
    X_max = X.max(axis=0)
    X_normalized = (X - X_min) / (X_max - X_min + 1e-10)
    X_train, X_test, y_train, y_test = train_test_split(
        X_normalized, y, test_size=0.3, random_state=42
    )
    data_registry[ds_name] = (X_train, X_test, y_train, y_test)



### ***Visualizing the datasets***
PCA reduction for easier visualization of the datasets in a two dimension space.

In [ ]:
for ds_name in selected_datasets:
    X_train, X_test, y_train, y_test = data_registry[ds_name]
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_train)

    plt.figure(figsize=(8, 6))
    sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=y_train, palette='viridis')
    plt.title(f"PCA Projection - {ds_name} (Fase 1)")
    plt.show()